In [3]:
import numpy as np

X = np.array([2.0, 4.0])
y = 6.0
lr = 0.01
EPOCHS = 200

W1 = np.array([[0.21, 0.77],
               [0.24, 0.56]])
B1 = np.array([0.71, 0.45])
W2 = np.array([[0.98, 0.65],
               [0.31, 0.52]])
B2 = np.array([0.88, 0.22])
W3 = np.array([0.48, 0.17])
B3 = 0.51


def relu(z):
    return np.maximum(0.0, z)


def relu_grad(z):
    return (z > 0).astype(float)


def forward(x, W1, B1, W2, B2, W3, B3):
    Z1 = W1 @ x + B1
    A1 = relu(Z1)
    Z2 = W2 @ A1 + B2
    A2 = relu(Z2)
    Z3 = W3 @ A2 + B3
    return Z1, A1, Z2, A2, Z3


def backward(x, target, W1, B1, W2, B2, W3, B3):
    Z1, A1, Z2, A2, Z3 = forward(x, W1, B1, W2, B2, W3, B3)
    y_hat = Z3
    J = 0.5 * (y_hat - target) ** 2

    dJ_dyhat = y_hat - target
    dJ_dZ3 = dJ_dyhat

    dW3 = dJ_dZ3 * A2
    dB3 = dJ_dZ3

    dJ_dA2 = dJ_dZ3 * W3
    dJ_dZ2 = dJ_dA2 * relu_grad(Z2)
    dW2 = np.outer(dJ_dZ2, A1)
    dB2 = dJ_dZ2

    dJ_dA1 = W2.T @ dJ_dZ2
    dJ_dZ1 = dJ_dA1 * relu_grad(Z1)
    dW1 = np.outer(dJ_dZ1, x)
    dB1 = dJ_dZ1

    cache = dict(Z1=Z1, A1=A1, Z2=Z2, A2=A2, Z3=Z3, y_hat=y_hat, J=J,
                 dJ_dyhat=dJ_dyhat, dJ_dA2=dJ_dA2, dJ_dZ2=dJ_dZ2,
                 dJ_dA1=dJ_dA1, dJ_dZ1=dJ_dZ1)
    grads = dict(W1=dW1, B1=dB1, W2=dW2, B2=dB2, W3=dW3, B3=dB3)
    return cache, grads


def names_values(W1, B1, W2, B2, W3, B3):
    return [
        ("W11", W1[0, 0]), ("W12", W1[0, 1]), ("W13", W1[1, 0]), ("W14", W1[1, 1]),
        ("B11", B1[0]), ("B12", B1[1]),
        ("W21", W2[0, 0]), ("W22", W2[0, 1]), ("W23", W2[1, 0]), ("W24", W2[1, 1]),
        ("B21", B2[0]), ("B22", B2[1]),
        ("W31", W3[0]), ("W32", W3[1]),
        ("B3", B3),
    ]


def grad_list(g):
    return [
        g["W1"][0, 0], g["W1"][0, 1], g["W1"][1, 0], g["W1"][1, 1],
        g["B1"][0], g["B1"][1],
        g["W2"][0, 0], g["W2"][0, 1], g["W2"][1, 0], g["W2"][1, 1],
        g["B2"][0], g["B2"][1],
        g["W3"][0], g["W3"][1],
        g["B3"],
    ]


def sgd_step(params, grads, lr):
    W1, B1, W2, B2, W3, B3 = params
    return (W1 - lr * grads["W1"], B1 - lr * grads["B1"],
            W2 - lr * grads["W2"], B2 - lr * grads["B2"],
            W3 - lr * grads["W3"], B3 - lr * grads["B3"])


print("\n1. Initial parameters")
for n, v in names_values(W1, B1, W2, B2, W3, B3):
    print(f"  {n} = {v:.6f}")

cache, grads = backward(X, y, W1, B1, W2, B2, W3, B3)

print("\n2. Forward propagation")
print(f"  Z11 = {cache['Z1'][0]:.6f}   A11 = {cache['A1'][0]:.6f}")
print(f"  Z12 = {cache['Z1'][1]:.6f}   A12 = {cache['A1'][1]:.6f}")
print(f"  Z21 = {cache['Z2'][0]:.6f}   A21 = {cache['A2'][0]:.6f}")
print(f"  Z22 = {cache['Z2'][1]:.6f}   A22 = {cache['A2'][1]:.6f}")
print(f"  Z3 = y_hat = {cache['y_hat']:.6f}")

print("\n3. Loss")
print(f"  J = 0.5*(y_hat - y)^2 = 0.5*({cache['y_hat']:.6f} - {y})^2 = {cache['J']:.6f}")

print("\n4. Backpropagation")
print(f"  dJ/dy_hat = {cache['dJ_dyhat']:.6f}")
print(f"  dJ/dA21 = {cache['dJ_dA2'][0]:.6f}   dJ/dA22 = {cache['dJ_dA2'][1]:.6f}")
print(f"  dJ/dZ21 = {cache['dJ_dZ2'][0]:.6f}   dJ/dZ22 = {cache['dJ_dZ2'][1]:.6f}")
print(f"  dJ/dA11 = {cache['dJ_dA1'][0]:.8f}   dJ/dA12 = {cache['dJ_dA1'][1]:.8f}")
print(f"  dJ/dZ11 = {cache['dJ_dZ1'][0]:.8f}   dJ/dZ12 = {cache['dJ_dZ1'][1]:.8f}")

print("\n5. All gradients")
for (n, _), g in zip(names_values(W1, B1, W2, B2, W3, B3), grad_list(grads)):
    print(f"  dJ/d{n} = {g:.8f}")

new_params = sgd_step((W1, B1, W2, B2, W3, B3), grads, lr)

print(f"\n6. Parameter updates (lr = {lr})")
old_vals = names_values(W1, B1, W2, B2, W3, B3)
new_vals = names_values(*new_params)
for (n, old), (_, new), g in zip(old_vals, new_vals, grad_list(grads)):
    print(f"  {n}_new = {old:.6f} - {lr}*({g:.8f}) = {new:.8f}")

*_, y_hat_after = forward(X, *new_params)
J_after = 0.5 * (y_hat_after - y) ** 2
print("\n7. After the first step")
print(f"  y_hat = {y_hat_after:.6f}   J = {J_after:.6f}")

params = (W1, B1, W2, B2, W3, B3)
for epoch in range(EPOCHS):
    _, g = backward(X, y, *params)
    params = sgd_step(params, g, lr)

print("\n" + line)
print(f"FINAL RESULT AFTER {EPOCHS} EPOCHS")
print(line)

*_, y_final = forward(X, *params)
J_final = 0.5 * (y_final - y) ** 2
for n, v in names_values(*params):
    print(f"  {n} = {v:.6f}")
print(f"\n  y_hat = {y_final:.6f}   (target y = {y})")
print(f"  J = {J_final:.10f}")


1. Initial parameters
  W11 = 0.210000
  W12 = 0.770000
  W13 = 0.240000
  W14 = 0.560000
  B11 = 0.710000
  B12 = 0.450000
  W21 = 0.980000
  W22 = 0.650000
  W23 = 0.310000
  W24 = 0.520000
  B21 = 0.880000
  B22 = 0.220000
  W31 = 0.480000
  W32 = 0.170000
  B3 = 0.510000

2. Forward propagation
  Z11 = 4.210000   A11 = 4.210000
  Z12 = 3.170000   A12 = 3.170000
  Z21 = 7.066300   A21 = 7.066300
  Z22 = 3.173500   A22 = 3.173500
  Z3 = y_hat = 4.441319

3. Loss
  J = 0.5*(y_hat - y)^2 = 0.5*(4.441319 - 6.0)^2 = 1.214743

4. Backpropagation
  dJ/dy_hat = -1.558681
  dJ/dA21 = -0.748167   dJ/dA22 = -0.264976
  dJ/dZ21 = -0.748167   dJ/dZ22 = -0.264976
  dJ/dA11 = -0.81534603   dJ/dA12 = -0.62409587
  dJ/dZ11 = -0.81534603   dJ/dZ12 = -0.62409587

5. All gradients
  dJ/dW11 = -1.63069206
  dJ/dW12 = -3.26138412
  dJ/dW13 = -1.24819174
  dJ/dW14 = -2.49638349
  dJ/dB11 = -0.81534603
  dJ/dB12 = -0.62409587
  dJ/dW21 = -3.14978256
  dJ/dW22 = -2.37168901
  dJ/dW23 = -1.11554799
  dJ/dW2